# Datamine GRTON Process Example

This notebook demonstrates how to configure and run the **`grton`** process wrapper in `dmstudio`.

## Process Description

## GRTON

Calculate grades and tonnages using results from a Gaussian anamorphosis.

**Note** : The process [GAUSAN](<gausan.md>) must have been run previously to create a file of the polynomial coefficients.

### Input Files:

* **polyno**:
  Input file containing polynomial coefficients. This file is created as output by
  process **GAUSAN**. The following fields are required:
  Required=No

### Output Files:

* **qtn** (Undefined):
  Compulsory output file containing grade and tonnage values for different cut-offs.
  Change of support calculations are NOT used. It includes the following explicit fields:
  Required=Yes

* **qtr** (Undefined):
  Optional output file containing grade and tonnage values for different cut-offs. Change
  of support calculations are used. It includes the following explicit fields:
  Required=No

### Fields:

### Parameters:

* **support**:
  Change of support flag: Options: 0: Support change calculations not used; 1: Support
  change calculations used The default value is (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **blockvar**:
  The block variance. This parameter is compulsory if a change of support is required; ie
  if SUPPORT=1.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **maxiter**:
  Maximum number of iterations. The default value is (100).
  Range=Undefined
  Values=Undefined
  Default=100
  Required=No

* **print**:
  Controls the amount of text displayed: The default value is (0). Options: 0: Minimum
  output to screen; 1: As 0 + grade/tonnage results; 2: As 1 + display for each iteration
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('grton')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute grton
print("Running grton...")
dm_cmd.grton(
    polyno_i='optional',  # required
    qtn_o='t_grton_out',  # required
    # qtr_o='t_grton_out',  # optional
    # support_p=0,  # optional
    # blockvar_p='optional',  # optional
    # maxiter_p=100,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("grton execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_grton_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")